# Part II — DeSE2026 revision: nested validation, paired bootstrap, full-information baseline, SMS-free model, repeated splits

This notebook reproduces every number in the revised conference paper *From Prediction to Action: Policy-Constrained Machine Learning for Lead Qualification Under Limited Sales Capacity* (DeSE2026, paper #1571378349). It is driven by `revision_experiments.py`, which re-implements the dissertation pipeline (Part I notebook, unchanged) and adds the analyses requested by the reviewers.

**Pinned versions.** XGBoost 2.1.4–3.2.0 reproduce the submitted numbers (265/270/266 converters) exactly; 3.3.0 and later do not (262/269/264) because of changes in tree-construction numerics. This notebook pins XGBoost 3.2.0.

**Runtime.** Steps 1–5 take about 5 minutes on 2 CPUs. Step 6 (25 independently tuned splits × 2 specifications) takes about 45 minutes; set `RUN_REPEATED = True` to recompute, otherwise the stored `results/repeated_splits.csv` is loaded.

In [1]:
!pip install -q "xgboost==3.2.0" "scikit-learn==1.9.1" "pandas>=2.2" "numpy>=2" scipy
import sys, os, json, hashlib
import numpy as np, pandas as pd
REPO = "." if os.path.exists("revision_experiments.py") else "/content/drive/MyDrive/Masters_Constrained_Lead_Qualification"   # repo root, or Colab + Drive
DATA = f"{REPO}/Lead Scoring.csv" if os.path.exists(f"{REPO}/Lead Scoring.csv") else "/content/drive/MyDrive/Masters_Constrained_Lead_Qualification/Lead Scoring.csv"
sys.path.insert(0, REPO)
import revision_experiments as R
import xgboost, sklearn
print("xgboost", xgboost.__version__, "| scikit-learn", sklearn.__version__)
print("dataset sha256:", hashlib.sha256(open(DATA, "rb").read()).hexdigest())

xgboost 3.2.0 | scikit-learn 1.9.1
dataset sha256: 1426dffd94246b8c7f08d1080d9c04115a0c4b67eadcba1ed9d2fa10d3762802


## 1. Data and the SMS overlap (Review 1, point 4)

In [2]:
df, policy = R.load_and_clean(DATA)                                   # notebook-identical cleaning (global >40% rule)
df_fold, policy_fold = R.load_and_clean(DATA, drop_high_missing_globally=False)   # fold-wise rule, used by all nested runs
assert df.shape == (7553, 28), df.shape
from sklearn.model_selection import train_test_split
y = df["Converted"]; X = df.drop(columns=["Converted"])
X_tr, X_te, y_tr, y_te, P_tr, P_te = train_test_split(X, y, policy, test_size=0.2, random_state=42, stratify=y)
for name, P_, X_ in [("train", P_tr, X_tr), ("test", P_te, X_te)]:
    ct = pd.crosstab(P_["Last Activity"].eq("SMS Sent"), X_["Last Notable Activity"].eq("SMS Sent"),
                     rownames=["Last Activity = SMS Sent (P2)"], colnames=["Last Notable Activity = SMS Sent (in model)"])
    print(f"\n{name}:\n{ct}\n  share of P2 leads carrying the leaked feature: {ct.loc[True, True] / ct.loc[True].sum():.3f}"
          f"\n  share of leaked-feature leads that are P2: {ct.loc[True, True] / ct[True].sum():.3f}")


train:
Last Notable Activity = SMS Sent (in model)  False  True 
Last Activity = SMS Sent (P2)                            
False                                         4215      0
True                                           407   1420
  share of P2 leads carrying the leaked feature: 0.777
  share of leaked-feature leads that are P2: 1.000

test:
Last Notable Activity = SMS Sent (in model)  False  True 
Last Activity = SMS Sent (P2)                            
False                                         1042      0
True                                            98    371
  share of P2 leads carrying the leaked feature: 0.791
  share of leaked-feature leads that are P2: 1.000


## 2. Reproduction of the as-submitted configuration
Single 80/20 split, full-data cutoff, cap 40%, w 0.90, test-selected parameters. Expected under XGBoost ≤ 3.2.0: S1 265, S2 270, S3 266, test AUC 0.848.

In [3]:
from sklearn.metrics import roc_auc_score
Xr_tr, Xr_te = R.feature_frame(X_tr, "orig"), R.feature_frame(X_te, "orig")
xgb0, params0 = R.tune_xgb(Xr_tr, y_tr)
res0 = P_te.reset_index(drop=True).copy(); res0["actual"] = y_te.values; res0["ml_score"] = xgb0.predict_proba(Xr_te)[:, 1]
sys0, diag0 = R.rank_systems(res0, ["Working Professional"], ["SMS Sent"], 0.40, 0.90, float(df["Converted"].mean()))
print("test AUC", round(roc_auc_score(y_te, res0["ml_score"]), 4), "| params", params0)
print("converters / Precision@K:", R.precision_at_k(sys0, diag0["K"]))
print("diagnostics:", diag0)

test AUC 0.8482 | params {'model__subsample': 0.7, 'model__n_estimators': 100, 'model__max_depth': 2, 'model__learning_rate': 0.1, 'model__colsample_bytree': 0.8}
converters / Precision@K: {'S1': (265, 0.8774834437086093), 'S2': (270, 0.8940397350993378), 'S3': (266, 0.8807947019867549)}
diagnostics: {'K': 302, 'cap': 120, 'h1_removed': 0, 'pool': 1511, 'p1_in_pool': 92, 'p2_in_pool': 341, 'demoted_by_cutoff': 104, 's2_from_p1': 92, 's2_from_p2': 120, 's2_from_residual': 90}


## 3. Nested 60/20/20 runs, primary split (seed 42), both specifications
Engine = validation-AUC winner of LR vs XGBoost; signals by the pre-stated rule on training rows; cap and weight on validation Precision@K; test evaluated once. `nosms` = primary specification (Last Notable Activity removed from the initial model); `orig` = original feature set.

In [4]:
runs = {spec: R.nested_run(df_fold, policy_fold, 42, spec, tune=True) for spec in ["nosms", "orig"]}
keys = ["selected_engine", "val_auc_logit", "val_auc_xgb", "test_auc_engine", "val_auc_full", "test_auc_full", "p1", "p2",
        "p1_train_rate", "p1_train_n", "p2_train_rate", "p2_train_n", "cutoff", "cap", "blend_w", "fold_dropped_high_missing",
        "K", "S1_conv", "S2_conv", "S3_conv", "S4_conv", "S1_prec", "S2_prec", "S3_prec", "S4_prec", "diag_test", "cap_sens_test", "k_sens_test"]
for spec, r in runs.items():
    print(f"\n=== {spec} ===")
    for k in keys: print(f"  {k}: {r[k]}")
    print("  leakage audit:", r["leakage_audit"])


=== nosms ===
  selected_engine: XGBoost
  val_auc_logit: 0.8356228629939113
  val_auc_xgb: 0.8444259567387687
  test_auc_engine: 0.8036212539540326
  val_auc_full: 0.8953566400321809
  test_auc_full: 0.8650125249126913
  p1: ['Working Professional']
  p2: ['SMS Sent']
  p1_train_rate: 0.9032258064516129
  p1_train_n: 372
  p2_train_rate: 0.5855263157894737
  p2_train_n: 1368
  cutoff: 0.3979254027808431
  cap: 0.6
  blend_w: 0.8
  fold_dropped_high_missing: ['How did you hear about X Education', 'Lead Profile']
  K: 302
  S1_conv: 224
  S2_conv: 255
  S3_conv: 252
  S4_conv: 261
  S1_prec: 0.7417218543046358
  S2_prec: 0.8443708609271523
  S3_prec: 0.8344370860927153
  S4_prec: 0.8642384105960265
  diag_test: {'K': 302, 'cap': 181, 'h1_removed': 0, 'pool': 1511, 'p1_in_pool': 87, 'p2_in_pool': 261, 'demoted_by_cutoff': 189, 's2_from_p1': 87, 's2_from_p2': 181, 's2_from_residual': 34}
  cap_sens_test: {'0.2': 0.7549668874172185, '0.4': 0.8112582781456954, '0.6': 0.8443708609271523, '1

## 4. Paired bootstrap (Review 1 point 1, Review 2 point 3, Review 3 point 3)
Every system is applied to the same resample; the difference is computed within each resample. Intervals condition on the trained models and selected policies.

In [5]:
pb = {}
for spec, r in runs.items():
    pb[spec], raw = R.paired_bootstrap(r["_res_te"], r["p1"], r["p2"], r["cap"], r["blend_w"], r["cutoff"], n=2000)
    rows = [{"difference": k.split("_")[0], "metric": k.split("_")[1], **v} for k, v in pb[spec].items() if "-" in k]
    print(f"\n=== {spec} ===\n", pd.DataFrame(rows).round(4).to_string(index=False))


=== nosms ===
 difference metric    mean     lo95    hi95  p_gt0  p_ge0
     S2-S1   prec  0.0988   0.0464  0.1490 1.0000 1.0000
     S2-S1   conv 29.8240  14.0000 45.0000 1.0000 1.0000
     S2-S3   prec  0.0062  -0.0298  0.0397 0.5715 0.6505
     S2-S3   conv  1.8640  -9.0000 12.0000 0.5715 0.6505
     S3-S1   prec  0.0926   0.0430  0.1424 1.0000 1.0000
     S3-S1   conv 27.9600  13.0000 43.0000 1.0000 1.0000
     S2-S4   prec -0.0217  -0.0563  0.0099 0.0720 0.1150
     S2-S4   conv -6.5520 -17.0000  3.0000 0.0720 0.1150
     S4-S1   prec  0.1205   0.0762  0.1623 1.0000 1.0000
     S4-S1   conv 36.3760  23.0000 49.0000 1.0000 1.0000



=== orig ===
 difference metric    mean     lo95    hi95  p_gt0  p_ge0
     S2-S1   prec  0.0308   0.0033  0.0596 0.9825 0.9915
     S2-S1   conv  9.2875   1.0000 18.0000 0.9825 0.9915
     S2-S3   prec  0.0115  -0.0166  0.0397 0.7575 0.8205
     S2-S3   conv  3.4705  -5.0000 12.0000 0.7575 0.8205
     S3-S1   prec  0.0193  -0.0066  0.0464 0.9010 0.9405
     S3-S1   conv  5.8170  -2.0000 14.0000 0.9010 0.9405
     S2-S4   prec -0.0127  -0.0331  0.0066 0.0860 0.1390
     S2-S4   conv -3.8475 -10.0000  2.0000 0.0860 0.1390
     S4-S1   prec  0.0435   0.0165  0.0695 0.9990 0.9995
     S4-S1   conv 13.1350   4.9750 21.0000 0.9990 0.9995


## 5. Capacity and tier-cap sensitivity on the test partition (policy parameters held at validation-selected values)

In [6]:
for spec, r in runs.items():
    print(f"\n=== {spec}: Precision@K by capacity ===")
    print(pd.DataFrame(r["k_sens_test"]).T.round(3))
    print(f"=== {spec}: S2 Precision@K by tier cap ===")
    print(pd.Series(r["cap_sens_test"]).round(3).to_string())


=== nosms: Precision@K by capacity ===
        S1     S2     S3     S4
0.1  0.801  0.881  0.881  0.934
0.2  0.742  0.844  0.834  0.864
0.3  0.709  0.788  0.775  0.799
=== nosms: S2 Precision@K by tier cap ===
0.2    0.755
0.4    0.811
0.6    0.844
1.0    0.841

=== orig: Precision@K by capacity ===
        S1     S2     S3     S4
0.1  0.914  0.947  0.907  0.934
0.2  0.851  0.887  0.874  0.901
0.3  0.784  0.808  0.801  0.815
=== orig: S2 Precision@K by tier cap ===
0.2    0.861
0.4    0.887
0.6    0.891
1.0    0.871


## 6. Repeated independently tuned splits (Review 1 point 5, Review 3 point 3)
Each split re-runs the hyperparameter searches, engine selection, signal selection and policy selection on its own training/validation rows.

In [7]:
RUN_REPEATED = False
if RUN_REPEATED:
    rows = []
    for spec in ["orig", "nosms"]:
        for seed in range(25):
            r = R.nested_run(df_fold, policy_fold, seed, spec, tune=True)
            rows.append({k: r[k] for k in ["seed", "spec", "p1", "p2", "cap", "blend_w", "selected_engine", "test_auc_engine", "test_auc_full",
                                           "S1_conv", "S2_conv", "S3_conv", "S4_conv", "S1_prec", "S2_prec", "S3_prec", "S4_prec"]})
            print(seed, spec, r["S1_prec"], r["S2_prec"], r["S3_prec"], r["S4_prec"])
    rep = pd.DataFrame(rows); rep["p1"] = rep["p1"].astype(str); rep["p2"] = rep["p2"].astype(str)
    for a, b in [("S2", "S1"), ("S2", "S3"), ("S2", "S4"), ("S4", "S1")]:
        rep[f"{a}-{b}"] = rep[f"{a}_prec"] - rep[f"{b}_prec"]
else:
    rep = pd.read_csv(f"{REPO}/results/repeated_splits.csv")
for spec, g in rep.groupby("spec"):
    print(f"\n=== {spec} ===")
    print(g[["S1_prec", "S2_prec", "S3_prec", "S4_prec", "S2-S1", "S2-S3", "S2-S4", "S4-S1"]].agg(["mean", "std", "min", "max"]).T.round(4))
    print("S2 > S1:", int((g["S2-S1"] > 0).sum()), "| S2 > S3:", int((g["S2-S3"] > 0).sum()), "| S2 > S4:", int((g["S2-S4"] > 0).sum()), "of", len(g))
    print("cap chosen:", g["cap"].value_counts().to_dict(), "| w chosen:", g["blend_w"].value_counts().to_dict(), "| engine:", g["selected_engine"].value_counts().to_dict())


=== nosms ===
           mean     std     min     max
S1_prec  0.7878  0.0203  0.7417  0.8212
S2_prec  0.8499  0.0225  0.8079  0.8874
S3_prec  0.8548  0.0147  0.8278  0.8775
S4_prec  0.8746  0.0181  0.8245  0.9040
S2-S1    0.0621  0.0221  0.0298  0.1291
S2-S3   -0.0049  0.0197 -0.0331  0.0464
S2-S4   -0.0246  0.0169 -0.0563  0.0066
S4-S1    0.0868  0.0228  0.0530  0.1556
S2 > S1: 25 | S2 > S3: 9 | S2 > S4: 1 of 25
cap chosen: {0.6: 15, 0.4: 10} | w chosen: {0.8: 24, 0.9: 1} | engine: {'XGBoost': 25}

=== orig ===
           mean     std     min     max
S1_prec  0.8641  0.0163  0.8377  0.8907
S2_prec  0.8944  0.0125  0.8609  0.9139
S3_prec  0.8665  0.0167  0.8311  0.8940
S4_prec  0.8964  0.0127  0.8709  0.9172
S2-S1    0.0303  0.0131  0.0132  0.0530
S2-S3    0.0279  0.0120  0.0099  0.0497
S2-S4   -0.0020  0.0085 -0.0132  0.0232
S4-S1    0.0323  0.0110  0.0099  0.0497
S2 > S1: 25 | S2 > S3: 25 | S2 > S4: 8 of 25
cap chosen: {0.4: 25} | w chosen: {0.8: 10, 0.9: 9, 0.95: 6} | engine: {'XG

## 7. Class balancing and preprocessing isolation (Review 3 point 4)

In [8]:
import inspect
src = inspect.getsource(R.xgb_pipeline) + inspect.getsource(R.tune_logit) + inspect.getsource(R.nested_run)
assert "scale_pos_weight=spw" in src and "(y == 0).sum() / (y == 1).sum()" in src, "positive-class weight must come from the training labels passed in"
assert 'class_weight="balanced"' in src
assert "high_missing_columns(X_tr)" in src, "missingness rule must be decided on training rows"
print("Class weighting is computed from training labels only; no resampling anywhere; >40%-missing rule applied on training rows of each split.")

Class weighting is computed from training labels only; no resampling anywhere; >40%-missing rule applied on training rows of each split.


## 8. Check against the stored results file

In [9]:
S = json.load(open(f"{REPO}/results/summary.json"))
for spec in ["nosms", "orig"]:
    for k in ["S1_conv", "S2_conv", "S3_conv", "S4_conv"]:
        assert runs[spec][k] == S[f"primary_{spec}"][k], (spec, k, runs[spec][k], S[f"primary_{spec}"][k])
print("Primary-split converter counts match results/summary.json:", {s: [runs[s][k] for k in ["S1_conv", "S2_conv", "S3_conv", "S4_conv"]] for s in runs})
print("Provenance in summary.json:", {k: S["provenance"][k] for k in ["dataset_sha256", "xgboost", "sklearn", "bootstrap_n", "repeated_seeds"]})

Primary-split converter counts match results/summary.json: {'nosms': [224, 255, 252, 261], 'orig': [257, 268, 264, 272]}
Provenance in summary.json: {'dataset_sha256': '1426dffd94246b8c7f08d1080d9c04115a0c4b67eadcba1ed9d2fa10d3762802', 'xgboost': '3.2.0', 'sklearn': '1.9.1', 'bootstrap_n': 2000, 'repeated_seeds': 25}
